# ML Output Evaluator
A reusable notebook for evaluating machine learning model outputs.  
Computes classification and regression metrics, visualises results,  
and interprets findings in plain English.

**Author:** Abigael Cherotich  
**Dataset:** Breast Cancer Wisconsin (UCI)  
**Purpose:** AI Evaluation Engineering Portfolio — Project 1

## Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score
)

# Load the breast cancer dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='actual')

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Train a logistic regression classifier
model = LogisticRegression(max_iter=10000)
model.fit(X_train, y_train)

# Generate predictions — in a real eval role these are handed to you
y_pred = pd.Series(model.predict(X_test), name='predicted')
y_prob = pd.Series(model.predict_proba(X_test)[:, 1], name='probability')

print("Dataset : Breast Cancer Classification (Wisconsin)")
print(f"Features : {X.shape[1]} tumour measurements per patient")
print(f"Test set : {len(y_test)} records")
print(f"Classes  : 0 = Malignant  |  1 = Benign")
print(f"\nActual values sample    : {y_test.values[:5]}")
print(f"Predicted values sample : {y_pred.values[:5]}")

### What this notebook does
This notebook answers two clinical questions from the same breast cancer dataset:

- **Part 1 — Classification:** Is this tumour malignant or benign?
- **Part 2 — Regression:** How large is this tumour likely to be?

We evaluate model outputs for both question types using industry-standard
metrics, visualisations, and plain-English interpretation — the core workflow
of an AI evaluation specialist.

---
## Part 1 — Classification Evaluation
**Clinical question:** Is this tumour malignant or benign?

### Classification Metrics

In [ ]:
# Compute all core classification metrics
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
auc_roc   = roc_auc_score(y_test, y_prob)

print("=" * 45)
print("       CLASSIFICATION METRICS SUMMARY")
print("=" * 45)
print(f"  Accuracy   : {accuracy:.4f}")
print(f"  Precision  : {precision:.4f}")
print(f"  Recall     : {recall:.4f}")
print(f"  F1 Score   : {f1:.4f}")
print(f"  AUC-ROC    : {auc_roc:.4f}")
print("=" * 45)
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred,
      target_names=['Malignant', 'Benign']))

### Interpretation — Classification Metrics

#### Overall accuracy: 0.9766
The model correctly classified 97.66% of all 171 test cases.
In practical terms it made 4 errors out of 171 predictions.
For a medical screening tool this is strong performance.

#### Precision — Malignant: 0.97 / Benign: 0.98
Precision answers: *"When the model makes a prediction, how often is it right?"*

- When the model predicted malignant, it was correct 97% of the time
- When the model predicted benign, it was correct 98% of the time

Very few false alarms in either direction.

#### Recall — Malignant: 0.97 / Benign: 0.98
Recall answers: *"Out of all the real cases, how many did the model find?"*

- Out of 63 actual malignant tumours, the model caught 97% of them
- Out of 108 actual benign tumours, the model correctly cleared 98% of them

This means roughly 2 real malignant cases were missed.

#### F1 Score — Malignant: 0.97 / Benign: 0.98
F1 is the balance between precision and recall in one number.
Both classes scoring 0.97–0.98 tells us the model is not sacrificing
one for the other — it performs consistently well in both directions.

A large gap between precision and recall would signal the model is biased
toward one type of error. That gap does not exist here.

#### AUC-ROC: 0.9976
At every possible decision threshold — not just the default 0.5 cutoff —
the model correctly separates malignant from benign cases 99.76% of the time.

This is outstanding. It means even if we lower the threshold to catch more
malignant cases, the model still discriminates well between the two classes.

#### Class balance observation
Test set: 63 malignant (37%) / 108 benign (63%) — moderately imbalanced.
The model performs equally well on both classes despite this imbalance,
which confirms it has not simply learned to predict the majority class.

This is why AUC-ROC (0.9976) is the most trustworthy metric here —
it accounts for class imbalance in a way that raw accuracy does not.

#### Summary verdict
A well-performing classifier by every measure. The 4 errors it makes
are evenly distributed — no systematic bias toward either class.

**Recommended next step:** Run a threshold sensitivity analysis to determine
whether the 2 missed malignant cases can be recovered by adjusting the
decision threshold below 0.5.

### Confusion Matrix
Visual breakdown of correct and incorrect predictions by class.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted Malignant', 'Predicted Benign'],
    yticklabels=['Actual Malignant',    'Actual Benign'],
    linewidths=0.5, ax=ax
)
ax.set_title('Confusion Matrix — Breast Cancer Classification',
             fontsize=13, pad=15)
ax.set_ylabel('Actual Label', fontsize=11)
ax.set_xlabel('Predicted Label', fontsize=11)
fig.text(0.13, 0.01,
    "Top-left = True Negatives  |  Top-right = False Positives  |  "
    "Bottom-left = False Negatives  |  Bottom-right = True Positives",
    fontsize=8, color='gray')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"True Negatives  (correct malignant) : {cm[0][0]}")
print(f"False Positives (malignant → benign): {cm[0][1]}")
print(f"False Negatives (benign → malignant): {cm[1][0]}")
print(f"True Positives  (correct benign)    : {cm[1][1]}")

### Interpretation — Confusion Matrix

#### What the four numbers mean
The model made **4 errors** out of 171 predictions:

- **61 True Negatives** — malignant tumours correctly identified as malignant
- **106 True Positives** — benign tumours correctly identified as benign
- **2 False Positives** — malignant cases incorrectly labelled as benign
- **2 False Negatives** — benign cases incorrectly labelled as malignant

#### Which error type matters more — and why this question always matters
In cancer screening a **False Positive** means telling a malignant patient
they are benign — they go home without treatment. This is the more dangerous
error because it delays diagnosis and can be fatal.

A **False Negative** means flagging a healthy patient for further tests.
This causes unnecessary stress and cost, but the patient gets another
chance at a correct diagnosis.

This is why **Recall is the priority metric in medical AI** — you want to
catch every real case, even at the cost of some false alarms.

**Key evaluator principle:** a model cannot be judged as good or bad without
first asking which error type is more costly in context. The same accuracy
score means something different in cancer screening versus spam filtering.

### Threshold Sensitivity Analysis
Investigating whether adjusting the decision threshold recovers the 2 missed malignant cases.

In [ ]:
thresholds_to_test = [0.5, 0.45, 0.40, 0.35, 0.30]

print("Threshold Sensitivity Analysis")
print("=" * 62)
print(f"{'Threshold':<12} {'Recall':>8} {'Precision':>10} "
      f"{'F1':>8} {'Missed malignant':>18}")
print("-" * 62)

for thresh in thresholds_to_test:
    y_pred_t = (y_prob >= thresh).astype(int)
    r = recall_score(y_test, y_pred_t)
    p = precision_score(y_test, y_pred_t)
    f = f1_score(y_test, y_pred_t)
    missed = int(((y_test == 0) & (y_pred_t == 1)).sum())
    print(f"  {thresh:<10} {r:>8.4f} {p:>10.4f} {f:>8.4f} {missed:>18}")

print("=" * 62)
print("\nDefault threshold = 0.5")
print("Lowering threshold → higher recall, lower precision")
print("Trade-off: more cancers caught vs more false alarms")

### Interpretation — Threshold Sensitivity

The default decision threshold for classification models is **0.5** —
the model predicts malignant only when it is more than 50% confident.

Lowering this threshold forces the model to flag more cases as malignant,
catching cases it was previously less than 50% confident about.
This directly addresses the 2 missed malignant cases from our confusion matrix.

**The trade-off:**
- Lower threshold = higher recall = more malignant cases caught
- Lower threshold = lower precision = more healthy patients sent for unnecessary tests

The right threshold is not a data science decision — it is a clinical policy
decision. The evaluator's job is to present this trade-off clearly so that
medical teams can make an informed choice.

**This is the complete evaluator workflow:**
1. Run metrics → identify the problem (2 missed malignant cases)
2. Investigate → threshold sensitivity analysis
3. Present trade-offs → let domain experts decide the right threshold

### ROC Curve
Measures the model's ability to separate classes across all possible decision thresholds.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='steelblue', lw=2,
        label=f'ROC Curve (AUC = {auc_roc:.4f})')
ax.plot([0, 1], [0, 1], color='gray', lw=1,
        linestyle='--', label='Random classifier (AUC = 0.50)')
ax.fill_between(fpr, tpr, alpha=0.08, color='steelblue')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate  (1 - Specificity)', fontsize=11)
ax.set_ylabel('True Positive Rate  (Recall / Sensitivity)', fontsize=11)
ax.set_title('ROC Curve — Breast Cancer Classification', fontsize=13, pad=15)
ax.legend(loc='lower right', fontsize=10)
ax.annotate('Perfect classifier\nwould reach this corner',
            xy=(0, 1), xytext=(0.15, 0.85),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"AUC-ROC Score: {auc_roc:.4f}")
print(f"\nInterpretation scale:")
print(f"  0.50 = random guessing — no better than a coin flip")
print(f"  0.70 = acceptable")
print(f"  0.80 = good")
print(f"  0.90 = excellent")
print(f"  0.99 = outstanding — verify no data leakage")
print(f"  1.00 = perfect — almost certainly a problem")
print(f"\nOur model: {auc_roc:.4f} — ", end="")
if auc_roc >= 0.99:
    print("outstanding — verify there is no data leakage")
elif auc_roc >= 0.9:
    print("excellent performance")
elif auc_roc >= 0.8:
    print("good performance")
else:
    print("acceptable — investigate further")

### Interpretation — ROC Curve

The ROC curve plots the True Positive Rate (Recall) against the False
Positive Rate at every possible decision threshold — not just the default 0.5.

The **dashed grey line** represents a random classifier (AUC = 0.50) —
a model that guesses blindly. Every point above that line represents
our model doing better than chance.

The **blue curve hugging the top-left corner** shows the model achieves
high recall while keeping false positives very low across all thresholds,
not just at the default cutoff.

**AUC-ROC interpretation scale:**
- 0.50 = random guessing, no predictive value
- 0.70 = acceptable
- 0.80 = good
- 0.90 = excellent
- 0.99 = outstanding — verify no data leakage
- 1.00 = perfect — almost certainly a problem

**Our model score: 0.9976 — outstanding separation between classes.**

#### Why AUC-ROC is more reliable than accuracy on imbalanced data
Accuracy can be misleading on imbalanced datasets. If 95% of patients
are benign, a model that always predicts benign gets 95% accuracy while
being completely useless for detecting cancer.

AUC-ROC measures discrimination ability across all thresholds — it cannot
be inflated by class imbalance alone. This makes it the preferred metric
for medical screening evaluation.

---
## Part 2 — Regression Evaluation
**Clinical question:** How large is this tumour likely to be?

We now use the same breast cancer dataset to predict a continuous value —
tumour size (mean radius in mm) — instead of a binary category.

### Regression Metrics

In [ ]:
# Same dataset — predicting tumour size (mean radius in mm)
cancer = load_breast_cancer()
X_all = pd.DataFrame(cancer.data, columns=cancer.feature_names)

y_reg = X_all['mean radius']           # target: tumour size
X_reg = X_all.drop(columns=['mean radius'])  # features: everything else

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42
)

reg_model = LinearRegression()
reg_model.fit(X_train_r, y_train_r)
y_pred_r = pd.Series(
    reg_model.predict(X_test_r), name='predicted_radius'
)

mae  = mean_absolute_error(y_test_r, y_pred_r)
mse  = mean_squared_error(y_test_r, y_pred_r)
rmse = np.sqrt(mse)
r2   = r2_score(y_test_r, y_pred_r)

print("=" * 55)
print("       REGRESSION METRICS SUMMARY")
print("       Predicting tumour size (mean radius in mm)")
print("=" * 55)
print(f"  MAE  (Mean Absolute Error)  : {mae:.4f}")
print(f"  MSE  (Mean Squared Error)   : {mse:.4f}")
print(f"  RMSE (Root Mean Sq. Error)  : {rmse:.4f}")
print(f"  R²   (R-Squared)            : {r2:.4f}")
print("=" * 55)
print(f"\nTarget variable (tumour mean radius):")
print(f"  Min  : {y_test_r.min():.2f} mm")
print(f"  Max  : {y_test_r.max():.2f} mm")
print(f"  Mean : {y_test_r.mean():.2f} mm")
print(f"\nMAE of {mae:.4f}  = predictions off by {mae:.2f}mm on average")
print(f"RMSE of {rmse:.4f} = {rmse:.2f}mm penalising large errors more heavily")

### Interpretation — Regression Metrics

#### The clinical question
Instead of *"is this tumour malignant or benign?"* we are now asking
*"how large is this tumour likely to be?"*

Tumour size (mean radius) ranges from **6.98mm to 25.22mm** in our test set,
with an average of **14.01mm**.

#### MAE — Mean Absolute Error: 0.0456 (0.05mm)
On average, the model's tumour size predictions are off by just 0.05mm.
On a scale of 6.98mm to 25.22mm this is less than 0.3% of the average
tumour size — negligible in a clinical context.

#### MSE — Mean Squared Error: 0.0049
MSE squares every error before averaging, punishing large errors
disproportionately. Our MSE of 0.0049 is extremely small, confirming
there are very few large errors — the model is consistently accurate,
not just accurate on average.

MSE is rarely interpreted on its own (the squared units are not intuitive).
Its main use is as the foundation for RMSE.

#### RMSE — Root Mean Squared Error: 0.0702 (0.07mm)
RMSE converts MSE back to the original unit (millimetres) by taking
the square root. It is always larger than MAE.

Our RMSE of 0.07mm is only marginally larger than MAE (0.05mm).

**Key evaluator insight:** when RMSE is close to MAE, the model's errors
are consistently small — there are no outlier predictions pulling the
average up. This is a sign of stable, reliable performance.

#### R² — R-Squared: 0.9996
The model explains 99.96% of the variation in tumour size. This is
near-perfect predictive power.

**R² interpretation scale:**
- Below 0.5  = poor
- 0.5 – 0.7  = moderate
- 0.7 – 0.9  = good
- Above 0.9  = excellent
- Above 0.99 = verify for data leakage

#### Important evaluator flag — results this strong need investigation
An R² of 0.9996 should always trigger a data leakage check in a real
evaluation role.

In this case the high score is explainable: mean radius is mathematically
correlated with other radius and area measurements in the same dataset
(mean area = π × radius²). The model is essentially reverse-engineering
the radius from related measurements — not truly predicting an unknown value.

**This is a critical evaluation lesson:** strong metrics do not always mean
a useful model. An evaluator must ask *"why is this score so high?"*
before declaring success — which is exactly the kind of finding that makes
an evaluator valuable to a research team.

### Regression Visualisation
Plotting predicted vs actual tumour sizes to identify where the model performs well and where it fails.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Predicted vs Actual
axes[0].scatter(y_test_r, y_pred_r, alpha=0.4,
                color='steelblue', s=20, label='Predictions')
min_val = float(min(y_test_r.min(), y_pred_r.min()))
max_val = float(max(y_test_r.max(), y_pred_r.max()))
axes[0].plot([min_val, max_val], [min_val, max_val],
             'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Tumour Radius (mm)', fontsize=11)
axes[0].set_ylabel('Predicted Tumour Radius (mm)', fontsize=11)
axes[0].set_title('Predicted vs Actual Tumour Size', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].text(0.05, 0.92, f'R² = {r2:.4f}\nRMSE = {rmse:.4f}mm',
             transform=axes[0].transAxes, fontsize=10,
             verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Plot 2: Residuals distribution
residuals = y_test_r.values - y_pred_r.values
axes[1].hist(residuals, bins=40, color='steelblue',
             alpha=0.7, edgecolor='white')
axes[1].axvline(x=0, color='red', linestyle='--',
                lw=2, label='Zero error')
axes[1].axvline(x=residuals.mean(), color='orange',
                linestyle='-', lw=2,
                label=f'Mean error: {residuals.mean():.4f}mm')
axes[1].set_xlabel('Prediction Error (mm)', fontsize=11)
axes[1].set_ylabel('Number of Predictions', fontsize=11)
axes[1].set_title('Distribution of Prediction Errors', fontsize=13)
axes[1].legend(fontsize=10)

plt.suptitle('Regression Evaluation — Breast Cancer Tumour Size',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('regression_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

residuals_df = pd.DataFrame({
    'actual'   : y_test_r.values,
    'predicted': y_pred_r.values,
    'error'    : residuals,
    'abs_error': abs(residuals)
}).sort_values('abs_error', ascending=False)

print("Top 5 largest prediction errors:")
print(residuals_df.head(5).round(4).to_string(index=False))
print(f"\nMedian absolute error : {residuals_df['abs_error'].median():.4f}mm")
print(f"95th percentile error : {residuals_df['abs_error'].quantile(0.95):.4f}mm")

### Interpretation — Regression Visualisation

#### What the scatter plot tells us
In the Predicted vs Actual plot each dot represents one tumour.
A perfect model places every dot exactly on the red dashed line.
Our dots cluster tightly along that line across the full size range
(6.98mm to 25.22mm), confirming consistent performance at all tumour
sizes — not just in the middle range.

#### What the residuals plot tells us
The residuals plot shows the distribution of prediction errors.
A well-behaved model produces errors that:
- Are centred at zero (no systematic bias)
- Follow a roughly normal bell shape
- Have no long tails pulling to one side

Our residuals are tightly centred near zero with a clean bell shape —
confirming no systematic over or under prediction.

#### The five largest errors in context
- Largest error: **0.4174mm** on a 20.18mm tumour = 2.1% miss
- 5th largest: **0.1776mm** on a 17.60mm tumour = 1.0% miss

Even the worst predictions are within 2.1% of the actual value.
In clinical measurement this level of precision is exceptional.

#### Median vs 95th percentile — the full error picture
- **Median absolute error: 0.0306mm** — half of all predictions are within 0.03mm
- **95th percentile: 0.1436mm** — 95% of all predictions are within 0.14mm

The 95th percentile being this tight means there are no catastrophic
outlier predictions hiding in the data.

---
## Final Summary Report

In [ ]:
print("=" * 57)
print("      ML OUTPUT EVALUATOR — FINAL SUMMARY")
print("      Dataset: Breast Cancer (Wisconsin)")
print("=" * 57)

print("\nPART 1 — CLASSIFICATION")
print(f"  Clinical question : Is this tumour malignant or benign?")
print(f"  Accuracy          : {accuracy:.4f}")
print(f"  Precision         : {precision:.4f}")
print(f"  Recall            : {recall:.4f}")
print(f"  F1 Score          : {f1:.4f}")
print(f"  AUC-ROC           : {auc_roc:.4f}")
print(f"  Errors            : 4 out of 171 predictions")
print(f"  Flag              : None — strong performance confirmed")

print("\nPART 2 — REGRESSION")
print(f"  Clinical question : How large is this tumour?")
print(f"  MAE               : {mae:.4f}mm average error")
print(f"  RMSE              : {rmse:.4f}mm")
print(f"  R²                : {r2:.4f}")
print(f"  95th pct error    : {residuals_df['abs_error'].quantile(0.95):.4f}mm")
print(f"  Flag              : Investigate feature correlation")
print(f"                      before production deployment")

print("\nEVALUATOR RECOMMENDATIONS")
print("  1. Run threshold sensitivity on classifier")
print("     — can we catch the 2 missed malignant cases?")
print("  2. Audit regression feature correlations")
print("     — R² of 0.9996 needs explanation before production")
print("  3. Test both models on external dataset")
print("     — confirm results hold on unseen real-world data")
print("=" * 57)

### Project Summary

#### What this notebook does
A reusable end-to-end ML evaluation notebook that takes model outputs
and produces a full evaluation report including:

- Classification metrics — accuracy, precision, recall, F1, AUC-ROC
- Confusion matrix with error type analysis
- Threshold sensitivity analysis
- ROC curve with threshold interpretation
- Regression metrics — MAE, MSE, RMSE, R²
- Predicted vs Actual visualisation
- Residuals distribution analysis
- Plain-English interpretation of every result

#### How to use it on your own model outputs
Replace the data loading section with your own:
- `y_test` — actual labels or values
- `y_pred` — model predicted labels
- `y_prob` — model predicted probabilities (classification only)

All metrics and visualisations compute automatically.

#### Skills demonstrated
- `sklearn` evaluation pipeline design
- `matplotlib` and `seaborn` visualisation
- Threshold sensitivity analysis and trade-off presentation
- Evaluation thinking: error type analysis, data leakage flags,
  context-aware metric selection
- Plain-English interpretation of technical results for stakeholders

#### Author
**Abigael Cherotich** — Data Analyst & AI Evaluation Specialist